# Bits e bytes

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx compila e roda a célula no seu computador. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Calcular quantos valores n bits guardam, e quantos bits um número precisa
- Converter tamanhos entre unidades SI (KB, MB) e unidades binárias (KiB, MiB)
- Diferenciar bits de bytes em velocidades e tamanhos de arquivo, e estimar um tempo de download
- Ler um número guardado em vários bytes, em ordem little-endian ou big-endian

Os **exercícios** deste notebook terminam com linhas `check(...)`: testes mínimos que imprimem ✓ quando o seu código está certo. Rode a próxima célula uma vez antes, para eles poderem usar o `check`.

In [ ]:
// shared: check(...) imprime ✓ ou ✗; os exercícios abaixo usam
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": esperado " + expected + ", veio " + actual);
    }
}

## Em resumo

Um **bit** é 0 ou 1, e um **byte** são 8 bits. Cada bit a mais dobra as possibilidades, então **n bits têm 2ⁿ valores**, de 0 a 2ⁿ − 1: um byte guarda de 0 a 255. No sentido contrário, um número precisa de tantos bits quantas vezes dá para dividi-lo ao meio até chegar a 0: 300 precisa de 9 (e 0 ainda precisa de 1 bit).

![Cada bit a mais dobra os padrões: 2, 4, 8](bit-patterns.svg)

**Cada bit a mais dobra as possibilidades:**

In [ ]:
for (int bits : new int[] {1, 2, 4, 8, 16, 32, 64}) {
    String valores = bits < 64 ? String.format("%,d", 1L << bits) : "18.446.744.073.709.551.616";
    IO.println(String.format("%2d bits: %s valores", bits, valores));
}

Os tamanhos usam dois tipos de prefixo. Os **prefixos SI** multiplicam por 1000: 1 KB = 1000 bytes, 1 MB = 1000² bytes. Os **prefixos binários** multiplicam por 1024: 1 KiB = 1024 bytes, 1 MiB = 1024² bytes. É por isso que um disco de 1 TB aparece com 931 "GB" no Windows: os mesmos bytes divididos por 1024³.

**O espaço "sumido" de um disco de 1 TB:**

In [ ]:
long bytes = 1_000_000_000_000L; // 1 TB, como diz a caixa
IO.println("em GB  (÷ 1000³): " + bytes / 1_000_000_000.0);
IO.println("em GiB (÷ 1024³): " + bytes / (1024.0 * 1024 * 1024));

Velocidades são em **bits** por segundo (Mbps, b minúsculo) e arquivos em **bytes** (MB, B maiúsculo): uma conexão de 100 Mbps transfere no máximo 12,5 MB por segundo.

Um número maior que 255 ocupa vários bytes, e eles podem ser guardados em duas ordens. O **big-endian** coloca o byte mais significativo primeiro, como escrevemos os números; o **little-endian** coloca o menos significativo primeiro. Os processadores Intel e ARM e muitos formatos de arquivo são little-endian; os protocolos de rede são big-endian.

### ✏️ Exercício: Um byte de um número

Devolva o byte número `index` de um int, contando a partir de 0 na ponta menos significativa: desloque para baixo e fique com 8 bits usando `& 0xFF`.

In [ ]:
int byteAt(int value, int index) {
    // TODO: desloque 8 × index, depois aplique a máscara
    return 0;
}

void main() {
    check("byte baixo de 300", byteAt(300, 0), 0x2C);
    check("byte alto de 300", byteAt(300, 1), 1);
    check("byte do topo", byteAt(0x12345678, 3), 0x12);
}

<details>
<summary>Ver uma resposta</summary>

```java
int byteAt(int value, int index) {
    return (value >> (8 * index)) & 0xFF;
}

void main() {
    check("byte baixo de 300", byteAt(300, 0), 0x2C);
    check("byte alto de 300", byteAt(300, 1), 1);
    check("byte do topo", byteAt(0x12345678, 3), 0x12);
}
```

</details>

| 300 = 0x012C | 1º byte | 2º byte |
|--------------|---------|---------|
| big-endian | `01` | `2C` |
| little-endian | `2C` | `01` |

#### 🎞️ Ordem dos bytes, passo a passo

![O número 300, 0x012C, guardado em big-endian como 01 2C e em little-endian como 2C 01, e lido de volta como 44 + 1 × 256](endianness-steps.pt-br.svg)

**As duas ordens de bytes, pelo Java:**

In [ ]:
int value = 300;
byte[] big = ByteBuffer.allocate(4).order(ByteOrder.BIG_ENDIAN).putInt(value).array();
byte[] little = ByteBuffer.allocate(4).order(ByteOrder.LITTLE_ENDIAN).putInt(value).array();
IO.println("big-endian:    " + HexFormat.ofDelimiter(" ").formatHex(big));
IO.println("little-endian: " + HexFormat.ofDelimiter(" ").formatHex(little));

### ✏️ Exercício: Leia dois bytes little-endian

Junte dois bytes de volta, o baixo primeiro, como num arquivo guardado em little-endian.

In [ ]:
int fromLittleEndian(int low, int high) {
    // TODO: combine os dois bytes
    return 0;
}

void main() {
    check("300", fromLittleEndian(0x2C, 0x01), 300);
    check("o maior", fromLittleEndian(255, 255), 65535);
    check("só o byte alto", fromLittleEndian(0, 1), 256);
}

<details>
<summary>Ver uma resposta</summary>

```java
int fromLittleEndian(int low, int high) {
    return low | (high << 8);
}

void main() {
    check("300", fromLittleEndian(0x2C, 0x01), 300);
    check("o maior", fromLittleEndian(255, 255), 65535);
    check("só o byte alto", fromLittleEndian(0, 1), 256);
}
```

</details>

## Leituras

- [Representing Numbers and Letters with Binary](https://www.youtube.com/watch?v=1GSjbWt0c9M) (Crash Course Computer Science #4, 11 min): A primeira metade: quantos valores 8 bits guardam, por que um kilobyte às vezes é 1000 e às vezes 1024 bytes, e o que querem dizer 32 bits e 64 bits.
- [Prefixes for binary multiples](https://physics.nist.gov/cuu/Units/binary.html) (NIST (U.S. National Institute of Standards and Technology), 4 min): A tabela dos prefixos binários (Ki, Mi, Gi) ao lado dos SI (k, M, G), e por que eles foram criados: para que um kilobyte possa ser exatamente 1000 bytes.
- [Little and Big Endian Mystery](https://www.geeksforgeeks.org/dsa/little-and-big-endian-mystery/) (GeeksforGeeks, 5 min): Como o mesmo número de 4 bytes fica na memória em cada ordem, e onde cada ordem é usada: processadores são em geral little-endian, protocolos de rede big-endian.

## Verifique

1. Quantos valores 10 bits guardam, e quantos bits o 1000 precisa?
2. Quantos bytes há em 2 MiB, e em 2 MB?
3. Que número os bytes little-endian `E8 03` guardam?

## Pratique

O notebook cobriu o básico. Agora aplique nos desafios da unidade, na barra lateral do Sphinx, onde o Submit roda todos os testes, inclusive os ocultos:

- **Quantos bits?** · Fácil
- **Unidades de armazenamento** · Médio
- **Tempo de download** · Médio
- **Lendo o cabeçalho de um BMP** · Difícil